# agml.ag_stats check (temporary)

Exercises every public function of `NASSCatalog` and `NASSDataLoader` on the real dataset.
Temporary notebook: do not commit it.

In [1]:
import os, pickle, tempfile
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

from agml.ag_stats import NASSDataLoader, NASSCatalog, TABLES, FORMATS

ROOT = "/group/jmearlesgrp/hkaman/dataset/nass-usda-full-data"
OUT = tempfile.mkdtemp(prefix="nass_check_")

/group/jmearlesgrp/hkaman/envs/agml-nass/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Loader setup

In [2]:
nass = NASSDataLoader(ROOT)
nass, nass.root

(<NASSDataLoader: /group/jmearlesgrp/hkaman/dataset/nass-usda-full-data/parquet>,
 '/group/jmearlesgrp/hkaman/dataset/nass-usda-full-data/parquet')

In [3]:
# Same loader from the environment variable.
os.environ["AGML_NASS_PATH"] = ROOT
NASSDataLoader().root == nass.root

True

In [4]:
# Every queryable table and its description.
pd.Series(NASSDataLoader.tables(), name="description")

yield              Yield at national, state, district, and county...
county_yield       County yield for the main crop variants, on 20...
area_production    Area planted, area harvested, and production (...
census             Census of Agriculture crop records, every five...
forecast           In-season yield forecasts by `forecast_month` ...
progress           Weekly crop progress by `stage`, as percent (s...
condition          Weekly crop condition, one column per rating (...
condition_long     Weekly crop condition, one row per rating (sta...
Name: description, dtype: object

In [5]:
FORMATS

('pandas', 'polars', 'arrow', 'records', 'csv', 'parquet')

In [6]:
# Column names and types of one table.
nass.schema("yield")

crop_id: string
commodity: string
class: string
practice: string
utilization: string
unit: string
agg_level: string
state_fips: string
state_alpha: string
asd_code: string
county_fips: string
county_fips_nass: string
county_name: string
is_other_counties: bool
year: int32
value: double
value_flag: string
cv_pct: double
value_t_ha: double
per_planted_acre: bool
load_time: timestamp[us]

In [7]:
# Pickle round trip (AgMLSerializable): only `root` is stored, the catalog is rebuilt.
clone = pickle.loads(pickle.dumps(nass))
clone.root == nass.root, len(clone.yield_data(state="IA", years=2020, level="state"))

(True, 13)

## 2. Catalog listings

In [8]:
cat = nass.catalog
len(cat.commodities()), cat.commodities()[:15]

(159,
 ['ALMONDS',
  'AMARANTH',
  'APPLES',
  'APRICOTS',
  'ARTICHOKES',
  'ASPARAGUS',
  'AVOCADOS',
  'BANANAS',
  'BARLEY',
  'BEANS',
  'BEETS',
  'BERRIES, OTHER',
  'BERRY TOTALS',
  'BLACKBERRIES',
  'BLUEBERRIES'])

In [9]:
cat.crops("corn")

,crop_id,commodity,class,practice,utilization,crop_label
0,407b68a8ae63,CORN,ALL CLASSES,ALL PRODUCTION PRACTICES,ALL UTILIZATION PRACTICES,CORN
1,c3d805b0e626,CORN,ALL CLASSES,BIOTECH,ALL UTILIZATION PRACTICES,CORN (BIOTECH)
2,0d60d988f714,CORN,ALL CLASSES,"BIOTECH, BT",ALL UTILIZATION PRACTICES,"CORN (BIOTECH, BT)"
3,bc9ebfaaa3a8,CORN,ALL CLASSES,"BIOTECH, HERBICIDE RESISTANT",ALL UTILIZATION PRACTICES,"CORN (BIOTECH, HERBICIDE RESISTANT)"
4,81f9e75f10e8,CORN,ALL CLASSES,"BIOTECH, STACKED GENE",ALL UTILIZATION PRACTICES,"CORN (BIOTECH, STACKED GENE)"
5,f3d600275f73,CORN,ALL CLASSES,IRRIGATED,ALL UTILIZATION PRACTICES,CORN (IRRIGATED)
6,6b2217b8ecc7,CORN,ALL CLASSES,NON-IRRIGATED,ALL UTILIZATION PRACTICES,CORN (NON-IRRIGATED)
7,9d9f23c859c7,CORN,ALL CLASSES,ALL PRODUCTION PRACTICES,FORAGE,"CORN, FORAGE"
8,f56a2f05c3bd,CORN,ALL CLASSES,ALL PRODUCTION PRACTICES,GRAIN,"CORN, GRAIN"
9,789e870f29d8,CORN,ALL CLASSES,IRRIGATED,GRAIN,"CORN, GRAIN (IRRIGATED)"


In [10]:
cat.states().head()

,state_fips,state_alpha,state_name,area_km2,centroid_lon,centroid_lat
0,01,AL,Alabama,133892.90,-86.83001,32.77881
1,02,AK,Alaska,1520598.73,-152.44197,63.78199
2,04,AZ,Arizona,295232.40,-111.66004,34.27701
3,05,AR,Arkansas,137782.04,-92.44218,34.89482
4,06,CA,California,409819.66,-119.47248,37.18624


In [11]:
cat.counties(state="IA").head()

,county_fips,state_fips,county_code,county_name,state_alpha,area_km2,centroid_lon,centroid_lat
0,19001,19,001,Adair,IA,1476.97,-94.47094,41.33066
1,19003,19,003,Adams,IA,1102.19,-94.69919,41.02896
2,19005,19,005,Allamakee,IA,1706.28,-91.37802,43.28413
3,19007,19,007,Appanoose,IA,1337.00,-92.86864,40.74311
4,19009,19,009,Audubon,IA,1148.29,-94.90578,41.68446


In [12]:
cat.districts(state="IA")

,state_fips,state_alpha,asd_code,asd_desc,asd_id
0,19,IA,10,NORTHWEST,1910
1,19,IA,20,NORTH CENTRAL,1920
2,19,IA,30,NORTHEAST,1930
3,19,IA,40,WEST CENTRAL,1940
4,19,IA,50,CENTRAL,1950
5,19,IA,60,EAST CENTRAL,1960
6,19,IA,70,SOUTHWEST,1970
7,19,IA,80,SOUTH CENTRAL,1980
8,19,IA,90,SOUTHEAST,1990
9,19,IA,98,"OTHER DISTRICTS, ALL COUNTIES",1998


In [13]:
# The catalog can also be used on its own.
NASSCatalog(nass.root).commodities()[:5]

['ALMONDS', 'AMARANTH', 'APPLES', 'APRICOTS', 'ARTICHOKES']

## 3. Catalog name resolution

In [14]:
[cat.resolve_states(v) for v in ["IA", "ia", "Iowa", "19", 19]], cat.resolve_states(["CA", "Illinois"])

([['19'], ['19'], ['19'], ['19'], ['19']], ['06', '17'])

In [15]:
cat.state_alphas(["19", "06"])

['IA', 'CA']

In [16]:
(cat.resolve_counties("Story", state="IA"),
 cat.resolve_counties("story county", state="Iowa"),
 cat.resolve_counties("19169"),
 cat.resolve_counties(["Yolo", "Fresno"], state="CA"))

(['19169'], ['19169'], ['19169'], ['06113', '06019'])

In [17]:
# Old NASS code for Shannon County, SD is mapped to its 2020 code (Oglala Lakota).
cat.resolve_counties("46113")

['46102']

In [18]:
first = cat.districts(state="IA").iloc[0]
print(first.to_dict())
(cat.resolve_districts(first["asd_id"]),
 cat.resolve_districts(first["asd_code"], state="IA"),
 cat.resolve_districts(first["asd_desc"].lower(), state="IA"))

{'state_fips': '19', 'state_alpha': 'IA', 'asd_code': '10', 'asd_desc': 'NORTHWEST', 'asd_id': '1910'}


(['1910'], ['1910'], ['1910'])

In [19]:
corn_all = cat.resolve_crops("corn")
corn_grain = cat.resolve_crops("corn", utilization="grain", practice="all production practices")
(len(corn_all), corn_grain,
 cat.resolve_crops("CORN, GRAIN"),
 cat.resolve_crops(["soybeans", "wheat"], crop_class="all classes", practice="all production practices",
                   utilization="all utilization practices"))

(26, ['f56a2f05c3bd'], ['f56a2f05c3bd'], ['8e3f7fc43681', '6256b3fe85b5'])

## 4. Error messages

In [20]:
for call in [
    lambda: cat.resolve_states("Iowaa"),
    lambda: cat.resolve_counties("Washington"),
    lambda: cat.resolve_counties("99999"),
    lambda: cat.resolve_districts("10"),
    lambda: cat.resolve_crops("cron"),
    lambda: cat.resolve_crops("corn", utilization="fiber"),
    lambda: nass.query("weather"),
    lambda: nass.progress(county="19169"),
    lambda: nass.county_yield(level="state"),
    lambda: nass.yield_data(where={"colour": "red"}),
    lambda: nass.yield_data(columns=["colour"]),
    lambda: nass.yield_data(format="xlsx"),
    lambda: nass.yield_data(format="csv"),
    lambda: NASSDataLoader("/no/such/dir"),
]:
    try:
        call()
        print("NO ERROR (unexpected)")
    except (ValueError, FileNotFoundError) as e:
        print(f"{type(e).__name__}: {e}")

ValueError: Unknown state 'Iowaa'. Did you mean one of: IOWA?
ValueError: County name 'Washington' exists in several states (AL, AR, CO, FL, GA, IA, ID, IL, IN, KS, KY, LA, MD, ME, MN, MO, MS, NC, NE, NY, OH, OK, OR, PA, RI, TN, TX, UT, VA, VT, WI). Pass `state` as well.
ValueError: Unknown county FIPS code '99999'.
ValueError: District code '10' is only unique within a state. Pass `state` as well.
ValueError: Unknown crop 'cron'. Did you mean one of: CORN, COTTON?
ValueError: Unknown utilization 'FIBER'.
ValueError: Unknown table 'weather'. Choose one of ['yield', 'county_yield', 'area_production', 'census', 'forecast', 'progress', 'condition', 'condition_long'].
ValueError: Table 'progress' has no county-level data, so `county` cannot be used with it.
ValueError: Table 'county_yield' has no state-level data, so `level` cannot be used with it.
ValueError: Table 'yield' has no column 'colour'.
ValueError: Table 'yield' has no column(s) ['colour'].
ValueError: Unknown format 'xlsx'. Cho

## 5. `yield` table: filters

In [21]:
# One county, one crop, a year range.
nass.yield_data(crop="corn", utilization="grain", state="IA", county="Story", years=(2015, 2024), level="county",
                columns=["year", "county_name", "value", "unit", "value_t_ha", "cv_pct"])

,year,county_name,value,unit,value_t_ha,cv_pct
0,2015,STORY,188.0,BU / ACRE,11.8003,NaN
1,2016,STORY,211.9,BU / ACRE,13.3005,NaN
2,2017,STORY,200.3,BU / ACRE,12.5724,NaN
3,2018,STORY,192.3,BU / ACRE,12.0702,NaN
4,2019,STORY,189.3,BU / ACRE,11.8819,NaN
5,2020,STORY,148.4,BU / ACRE,9.3147,2.6
6,2021,STORY,201.2,BU / ACRE,12.6288,1.9
7,2022,STORY,207.1,BU / ACRE,12.9992,2.5
8,2023,STORY,211.4,BU / ACRE,13.2691,3.2
9,2024,STORY,215.5,BU / ACRE,13.5264,2.6


In [22]:
# Year forms: single year, list, range object.
q = dict(crop="CORN, GRAIN", county="19169", level="county", columns=["year", "value"])
(nass.yield_data(years=2020, **q)["year"].tolist(),
 nass.yield_data(years=[1990, 2000, 2010], **q)["year"].tolist(),
 nass.yield_data(years=range(2021, 2024), **q)["year"].tolist())

([2020], [1990, 2000, 2010], [2021, 2022, 2023])

In [23]:
# Every level for one state and year (no `level` given).
nass.yield_data(crop="CORN, GRAIN", state="IA", years=2020)["agg_level"].value_counts()

agg_level
COUNTY    95
STATE      1
Name: count, dtype: int64

In [24]:
# State level, several states.
nass.yield_data(crop="CORN, GRAIN", state=["IA", "IL", "NE"], level="state", years=(2020, 2023),
                columns=["state_alpha", "year", "value", "unit"])

,state_alpha,year,value,unit
0,IA,2020,177.0,BU / ACRE
1,IA,2021,204.0,BU / ACRE
2,IA,2022,200.0,BU / ACRE
3,IA,2023,201.0,BU / ACRE
4,IL,2020,191.0,BU / ACRE
5,IL,2021,202.0,BU / ACRE
6,IL,2022,214.0,BU / ACRE
7,IL,2023,206.0,BU / ACRE
8,NE,2020,180.0,BU / ACRE
9,NE,2021,194.0,BU / ACRE


In [25]:
# National level.
nass.yield_data(crop="CORN, GRAIN", level="national", years=(2019, 2023), columns=["year", "value", "unit"])

,year,value,unit
0,2019,167.5,BU / ACRE
1,2020,171.4,BU / ACRE
2,2021,176.7,BU / ACRE
3,2022,173.4,BU / ACRE
4,2023,177.3,BU / ACRE


In [26]:
# District level. District yields in this data stop in 2019.
nass.yield_data(crop="CORN, GRAIN", district=first["asd_id"], level="district", years=(2015, 2019),
                columns=["state_alpha", "asd_code", "year", "value"])

,state_alpha,asd_code,year,value
0,IA,10,2015,202.5
1,IA,10,2016,205.5
2,IA,10,2017,203.4
3,IA,10,2018,188.4
4,IA,10,2019,195.3


In [27]:
# "Other (combined) counties" rows are dropped by default.
q = dict(crop="CORN, GRAIN", state="IA", level="county", years=2020)
kept = nass.yield_data(**q)
every = nass.yield_data(include_other_counties=True, **q)
len(kept), len(every), every["is_other_counties"].sum()

(95, 96, np.int64(1))

In [28]:
# `where` on any column, e.g. one unit only.
nass.yield_data(crop="wheat", state="KS", level="state", years=2023, where={"unit": "BU / ACRE"},
                columns=["crop_id", "year", "value", "unit"])

,crop_id,year,value,unit
0,6256b3fe85b5,2023,35.0,BU / ACRE
1,c66627e9fb28,2023,35.0,BU / ACRE


## 6. Other tables

In [29]:
# county_yield: the curated main-variant county view (filters by state_alpha internally).
nass.county_yield(state="CA", county="Yolo", years=(2018, 2020),
                  columns=["year", "commodity", "class", "utilization", "yield_t_ha", "qa_outlier"])

,year,commodity,class,utilization,yield_t_ha,qa_outlier
0,2018,CORN,ALL CLASSES,GRAIN,12.3464,False
1,2018,WHEAT,WINTER,ALL UTILIZATION PRACTICES,5.2389,False
2,2018,BARLEY,ALL CLASSES,ALL UTILIZATION PRACTICES,5.4554,False
3,2019,CORN,ALL CLASSES,GRAIN,12.2271,False
4,2019,WHEAT,WINTER,ALL UTILIZATION PRACTICES,3.1339,False
5,2020,WHEAT,WINTER,ALL UTILIZATION PRACTICES,7.3976,False


In [30]:
nass.area_production(crop="wheat", crop_class="all classes", state=["KS", "ND"], level="state",
                     statistic="production", years=(2020, 2024), columns=["state_alpha", "year", "value", "unit"])

,state_alpha,year,value,unit
0,KS,2020,281250000.0,BU
1,KS,2021,364000000.0,BU
2,KS,2022,244200000.0,BU
3,KS,2023,201250000.0,BU
4,KS,2024,307450000.0,BU
5,ND,2020,312587000.0,BU
6,ND,2021,196195000.0,BU
7,ND,2022,299900000.0,BU
8,ND,2023,306390000.0,BU
9,ND,2024,368285000.0,BU


In [31]:
# Several statistics at once.
nass.area_production(crop="CORN, GRAIN", county="Story", state="IA", years=2022,
                     statistic=["area_planted", "area_harvested", "production"],
                     columns=["year", "statistic", "value", "unit"])

,year,statistic,value,unit
0,2022,area_harvested,156100.0,ACRES
1,2022,production,32328000.0,BU


In [32]:
# Census of Agriculture: every five years. (The data has no CA almond records, so this uses Iowa corn grain.)
nass.census(crop="CORN, GRAIN", state="IA", level="state", statistic="area_harvested", where={"unit": "ACRES"},
            columns=["year", "statistic", "value", "unit"])

,year,statistic,value,unit
0,1997,area_harvested,11930542.0,ACRES
1,2002,area_harvested,11761392.0,ACRES
2,2007,area_harvested,13842282.0,ACRES
3,2012,area_harvested,13709408.0,ACRES
4,2017,area_harvested,12969645.0,ACRES
5,2022,area_harvested,12637070.0,ACRES


In [33]:
nass.forecast(crop="CORN, GRAIN", state="IA", years=2023, columns=["year", "forecast_month", "value", "unit"])

,year,forecast_month,value,unit
0,2023,AUG,203.0,BU / ACRE
1,2023,SEP,200.0,BU / ACRE
2,2023,OCT,199.0,BU / ACRE
3,2023,NOV,200.0,BU / ACRE


In [34]:
nass.progress(crop="soybeans", state="IL", years=2023, stage="planted", columns=["week_ending", "stage", "pct"])

,week_ending,stage,pct
0,2023-04-16,PLANTED,4.0
1,2023-04-23,PLANTED,15.0
2,2023-04-30,PLANTED,39.0
3,2023-05-07,PLANTED,66.0
4,2023-05-14,PLANTED,77.0
5,2023-05-21,PLANTED,85.0
6,2023-05-28,PLANTED,92.0
7,2023-06-04,PLANTED,96.0
8,2023-06-11,PLANTED,97.0


In [35]:
# Several stages, all three spellings accepted.
nass.progress(crop="soybeans", state="IL", years=2023, stage=["PCT PLANTED", "emerged"])["stage"].value_counts()

stage
EMERGED    10
PLANTED     9
Name: count, dtype: int64

In [36]:
nass.condition(crop="CORN", state="IA", years=2023,
               columns=["week_ending", "very_poor", "poor", "fair", "good", "excellent", "good_excellent"]).head()

,week_ending,very_poor,poor,fair,good,excellent,good_excellent
0,2023-05-28,0.0,2.0,21.0,65.0,12.0,77.0
1,2023-06-04,0.0,3.0,25.0,60.0,12.0,72.0
2,2023-06-11,1.0,3.0,26.0,58.0,12.0,70.0
3,2023-06-18,1.0,7.0,33.0,50.0,9.0,59.0
4,2023-06-25,2.0,10.0,32.0,48.0,8.0,56.0


In [37]:
nass.condition(long=True, crop="CORN", state="IA", years=2023, columns=["week_ending", "rating", "pct"]).head()

,week_ending,rating,pct
0,2023-05-28,EXCELLENT,12.0
1,2023-05-28,FAIR,21.0
2,2023-05-28,GOOD,65.0
3,2023-05-28,POOR,2.0
4,2023-05-28,VERY POOR,0.0


In [38]:
# Generic query() on any table.
nass.query("condition_long", crop="CORN", level="national", years=2023, where={"rating": "GOOD"},
           columns=["week_ending", "rating", "pct"]).head()

,week_ending,rating,pct
0,2023-05-28,GOOD,58.0
1,2023-06-04,GOOD,53.0
2,2023-06-11,GOOD,51.0
3,2023-06-18,GOOD,47.0
4,2023-06-25,GOOD,42.0


## 7. Output formats

In [39]:
q = dict(crop="CORN, GRAIN", state="IA", county="Story", years=(2020, 2023), columns=["year", "value"])
for fmt in ["pandas", "polars", "arrow", "records"]:
    try:
        r = nass.yield_data(format=fmt, **q)
        print(f"{fmt:8s} {type(r).__module__}.{type(r).__name__}  rows={len(r)}")
    except ImportError as e:
        print(f"{fmt:8s} skipped: {e}")
nass.yield_data(format="records", **q)[:2]

pandas   pandas.core.frame.DataFrame  rows=4


polars   polars.dataframe.frame.DataFrame  rows=4
arrow    pyarrow.lib.Table  rows=4


records  builtins.list  rows=4


[{'year': 2020, 'value': 148.4}, {'year': 2021, 'value': 201.2}]

In [40]:
for fmt in ["csv", "parquet"]:
    path = nass.yield_data(format=fmt, path=os.path.join(OUT, f"story_corn.{fmt}"), **q)
    print(path, os.path.getsize(path), "bytes")
pd.read_csv(os.path.join(OUT, "story_corn.csv"))

/tmp/nass_check_qps_bfc6/story_corn.csv 59 bytes
/tmp/nass_check_qps_bfc6/story_corn.parquet 786 bytes


,year,value
0,2020,148.4
1,2021,201.2
2,2022,207.1
3,2023,211.4


## 8. Timing on the full tables

In [41]:
%time _ = nass.yield_data(crop="CORN, GRAIN", county="Story", state="IA", years=(2000, 2024))
%time _ = nass.area_production(crop="CORN, GRAIN", level="county", years=2022)
%time _ = nass.yield_data(level="county", years=2022)

CPU times: user 1.06 s, sys: 551 ms, total: 1.61 s
Wall time: 830 ms


CPU times: user 2.45 s, sys: 243 ms, total: 2.69 s
Wall time: 1.38 s


CPU times: user 1.05 s, sys: 464 ms, total: 1.52 s
Wall time: 768 ms
